In [1]:
# 7-15-2026

In [ ]:
# for each cell in predicted transfer matrix, compute how much each branch in the embedding pair contributed to greatest euclidean distance
# see which branch correlates highest w transfer performance

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [5]:
embeddings = pd.read_csv("domain_embeddings_v4.csv")
embeddings.head()

,domain_id,e_0,e_1,e_2,e_3,e_4,e_5,e_6,e_7,e_8,src_bias,tgt_bias
0,0,0.044920,-0.140614,-0.092586,0.038657,0.091209,0.044721,-0.177195,0.017328,0.141784,0.204292,-0.017042
1,1,0.044615,-0.182949,-0.102645,0.036313,0.095344,0.107567,-0.248821,-0.000363,0.227570,0.176388,0.026637
2,2,0.065594,-0.110857,-0.039893,0.000854,0.076925,0.008618,-0.111971,-0.082349,0.096454,0.221703,-0.044998
3,4,0.081379,-0.068263,-0.103002,-0.023489,0.136152,0.028229,-0.148741,-0.059863,0.195213,0.231726,-0.063666
4,5,0.058112,-0.131804,-0.096756,0.065104,0.096903,0.066202,-0.164116,0.035039,0.129189,0.204279,-0.007497


In [6]:
predicted_T4 = pd.read_csv("../training/predicted_transfer_matrix_v4.csv")
predicted_T4.set_index("Unnamed: 0", inplace=True)
predicted_T4.index.name = None
predicted_T4.index = predicted_T4.index.astype(int)
predicted_T4.columns = predicted_T4.columns.astype(int)
predicted_T4.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.187250,0.212261,0.136024,0.118177,0.194689,0.124477,0.138357,0.197033,0.181316,0.114335,...,0.181668,0.168882,0.138788,0.267368,0.151536,0.077980,0.128121,0.139860,0.137399,0.144191
1,0.140677,0.203025,0.067787,0.072078,0.145408,0.078730,0.069400,0.167576,0.121355,0.067618,...,0.125502,0.108896,0.072622,0.215240,0.092988,0.043839,0.057884,0.101510,0.073914,0.115734
2,0.181390,0.184737,0.176705,0.135895,0.185064,0.153520,0.169430,0.184889,0.172432,0.129762,...,0.190400,0.188690,0.175312,0.261902,0.166325,0.079322,0.137719,0.183149,0.156938,0.121633
4,0.192235,0.217720,0.164587,0.168061,0.195178,0.172336,0.160411,0.202663,0.171474,0.143325,...,0.186857,0.186584,0.173756,0.258685,0.161375,0.114706,0.126835,0.207971,0.173703,0.141553
5,0.185131,0.207434,0.130140,0.111562,0.196782,0.113636,0.131514,0.190359,0.181921,0.119544,...,0.176199,0.160116,0.130384,0.268153,0.144690,0.076487,0.130301,0.127530,0.131348,0.137746


In [7]:
# align embeddings to the same domain order as the transfer matrix
active_ids = predicted_T4.index.tolist()

embeddings_indexed = embeddings.set_index("domain_id").loc[active_ids]

In [ ]:
# just the 9 embedding dims, drop src_bias/tgt_bias for this analyss
embed_cols = [c for c in embeddings_indexed.columns if c.startswith("e_")]
domain_embeds = embeddings_indexed[embed_cols].values
domain_embeds.shape

(34, 9)

In [9]:
# branch dim ranges, order is atm, ground, fire
branch_slices = {
    "atmospheric": slice(0, 3),
    "ground": slice(3, 6),
    "fire": slice(6, 9),
}
branch_names = list(branch_slices.keys())

In [10]:
# pairwise diff grid, shape (34, 34, 9)
diff = domain_embeds[:, None, :] - domain_embeds[None, :, :]
sq = diff ** 2

# per branch squared distance, shape (34, 34, 3), this is the rgb grid
branch_sq_grid = np.stack(
    [sq[:, :, s].sum(axis=-1) for s in branch_slices.values()],
    axis=-1
)

In [28]:
total_direct = sq.sum(axis=-1)
print("max decomposition error:", np.abs(branch_sq_grid.sum(axis=-1) - total_direct).max())

max decomposition error: 2.7755575615628914e-17


In [ ]:
# normalize per cell so the 3 channels sum to 1, like a photo
branch_frac_grid = branch_sq_grid / branch_sq_grid.sum(axis=-1, keepdims=True)
# error bc diagonal

C:\Users\Yash\AppData\Local\Temp\ipykernel_29760\473185250.py:2: RuntimeWarning: invalid value encountered in divide
  branch_frac_grid = branch_sq_grid / branch_sq_grid.sum(axis=-1, keepdims=True)


In [13]:
# which branch correlates highest with transfer performance
flat_targets = predicted_T4.loc[active_ids, active_ids].values.flatten()

for k, name in enumerate(branch_names):
    branch_dist_flat = branch_sq_grid[:, :, k].flatten()
    corr, _ = spearmanr(-branch_dist_flat, flat_targets)
    print(f"{name} branch sq dist vs transfer, spearman: {corr:.4f}")

atmospheric branch sq dist vs transfer, spearman: 0.5087
ground branch sq dist vs transfer, spearman: 0.4621
fire branch sq dist vs transfer, spearman: 0.4721


In [ ]:
# off-diagonal mask, same shape as branch_sq_grid[:,:,0]
n = len(active_ids)
off_diag_mask = ~np.eye(n, dtype=bool)

flat_targets_od = predicted_T4.loc[active_ids, active_ids].values[off_diag_mask]

for k, name in enumerate(branch_names):
    branch_dist_od = branch_sq_grid[:, :, k][off_diag_mask]
    corr, _ = spearmanr(-branch_dist_od, flat_targets_od)
    print(f"{name} branch sq dist vs transfer, off-diag spearman: {corr:.4f}")
# lowered bc on diag i=i has high corr (transfer<->embedding distance)

atmospheric branch sq dist vs transfer, off-diag spearman: 0.4891
ground branch sq dist vs transfer, off-diag spearman: 0.4396
fire branch sq dist vs transfer, off-diag spearman: 0.4497


In [29]:
# full 9-dim embedding distance vs transfer, off-diagonal only
total_dist_od = total_direct[off_diag_mask]  # total_direct was computed earlier as sq.sum(axis=-1)

full_corr, _ = spearmanr(-total_dist_od, flat_targets_od)
print(f"full embedding dist sq vs transfer, off-diag spearman: {full_corr:.4f}")

full embedding dist sq vs transfer, off-diag spearman: 0.6709


In [15]:
rows = []
for a, i in enumerate(active_ids):
    for b, j in enumerate(active_ids):
        if i == j:
            continue # ignore diag bc distances are just 0
        rows.append({
            "i": i,
            "j": j,
            "atmo_dist": branch_sq_grid[a, b, 0],
            "ground_dist": branch_sq_grid[a, b, 1],
            "fire_dist": branch_sq_grid[a, b, 2],
            "transfer": predicted_T4.loc[i, j],
        })

In [27]:
branch_dist_df = pd.DataFrame(rows)
branch_dist_df.head()

,i,j,atmo_dist,ground_dist,fire_dist,transfer
0,0,1,0.001894,0.003972,0.012802,0.212261
1,0,2,0.004089,0.002937,0.016244,0.136024
2,0,4,0.006672,0.006154,0.009623,0.118177
3,0,5,0.000269,0.001193,0.000643,0.194689
4,0,6,0.004144,0.005016,0.018330,0.124477


In [30]:
branch_dist_df.to_csv("branch_dist_df.csv", index=False)